In [1]:
import numpy as np
import matplotlib.pyplot as plt
import string
from sklearn.model_selection import train_test_split

In [2]:
input_files = [
  'edgar_allan_poe.txt',
  'robert_frost.txt',
]

In [3]:
!head edgar_allan_poe.txt

LO! Death hath rear'd himself a throne
In a strange city, all alone,
Far down within the dim west
Where the good, and the bad, and the worst, and the best,
Have gone to their eternal rest.
 
There shrines, and palaces, and towers
Are not like any thing of ours
Oh no! O no! ours never loom
To heaven with that ungodly gloom!


In [4]:
!head robert_frost.txt

Two roads diverged in a yellow wood,
And sorry I could not travel both
And be one traveler, long I stood
And looked down one as far as I could
To where it bent in the undergrowth; 

Then took the other, as just as fair,
And having perhaps the better claim
Because it was grassy and wanted wear,
Though as for that the passing there


2 empty list  one to hold text and another to label
enumerate guves index
rstrip - remove newline

In [5]:
# collect data into lists
input_texts = []
labels = []

for label, f in enumerate(input_files):
  print(f"{f} corresponds to label {label}")

  for line in open(f):
    line = line.rstrip().lower()
    if line:
      # remove punctuation
      line = line.translate(str.maketrans('', '', string.punctuation))

      input_texts.append(line)
      labels.append(label)

edgar_allan_poe.txt corresponds to label 0
robert_frost.txt corresponds to label 1


In [6]:
train_text, test_text, Ytrain, Ytest = train_test_split(input_texts, labels)

In [7]:
len(Ytrain), len(Ytest)

(1615, 539)

In [8]:
train_text[:5]

['disturbed the slumbering village street',
 'if the bones liked the attic let them have it',
 'cold as a spring as yet so near its source',
 'im stark he drew his passport',
 'and the shadow of thy perfect bliss']

In [9]:
Ytrain[:5]

[1, 1, 1, 1, 0]

In [10]:
idx = 1
word2idx = {'<unk>': 0}

In [11]:
# populate word2idx
for text in train_text:
    tokens = text.split()
    for token in tokens:
      if token not in word2idx:
        word2idx[token] = idx
        idx += 1

In [12]:
word2idx

{'<unk>': 0,
 'disturbed': 1,
 'the': 2,
 'slumbering': 3,
 'village': 4,
 'street': 5,
 'if': 6,
 'bones': 7,
 'liked': 8,
 'attic': 9,
 'let': 10,
 'them': 11,
 'have': 12,
 'it': 13,
 'cold': 14,
 'as': 15,
 'a': 16,
 'spring': 17,
 'yet': 18,
 'so': 19,
 'near': 20,
 'its': 21,
 'source': 22,
 'im': 23,
 'stark': 24,
 'he': 25,
 'drew': 26,
 'his': 27,
 'passport': 28,
 'and': 29,
 'shadow': 30,
 'of': 31,
 'thy': 32,
 'perfect': 33,
 'bliss': 34,
 'dont': 35,
 'donkeys': 36,
 'ears': 37,
 'suggest': 38,
 'we': 39,
 'shake': 40,
 'our': 41,
 'own': 42,
 'serenest': 43,
 'skies': 44,
 'continually': 45,
 'swollen': 46,
 'tight': 47,
 'buried': 48,
 'under': 49,
 'snow': 50,
 'how': 51,
 'are': 52,
 'you': 53,
 'neighbour': 54,
 'just': 55,
 'man': 56,
 'after': 57,
 'was': 58,
 'i': 59,
 'knew': 60,
 'good': 61,
 'reason': 62,
 'could': 63,
 'not': 64,
 'say': 65,
 'nose': 66,
 'is': 67,
 'same': 68,
 'sos': 69,
 'chin': 70,
 'mean': 71,
 'miles': 72,
 'from': 73,
 'that': 74,
 'mor

In [13]:
len(word2idx)

2530

index a and pi
2 empty list
one line of poem

In [14]:
# convert data into integer format
train_text_int = []
test_text_int = []

for text in train_text:
  tokens = text.split()
  line_as_int = [word2idx[token] for token in tokens]
  train_text_int.append(line_as_int)

for text in test_text:
  tokens = text.split()
  line_as_int = [word2idx.get(token, 0) for token in tokens]
  test_text_int.append(line_as_int)

result is list of list of integer

In [15]:
train_text_int[100:105]

[[144, 356, 382, 209, 328, 205, 58, 383],
 [384, 295, 2, 385, 386],
 [64, 182, 387, 49, 388, 56, 29, 389],
 [390, 65, 391, 392, 151, 134, 276, 298, 393, 74],
 [2, 394, 395, 356, 396, 295, 222, 397]]

In [16]:
# initialize A and pi matrices - for both classes
V = len(word2idx)

A0 = np.ones((V, V))
pi0 = np.ones(V)

A1 = np.ones((V, V))
pi1 = np.ones(V)

TExt to the particular class, a and pi as object
text as int- each line of the poem represented as list of int

In [17]:
# compute counts for A and pi
def compute_counts(text_as_int, A, pi):
  for tokens in text_as_int:
    last_idx = None
    for idx in tokens:
      if last_idx is None:
        # it's the first word in a sentence
        pi[idx] += 1
      else:
        # the last word exists, so count a transition
        A[last_idx, idx] += 1

      # update last idx
      last_idx = idx


compute_counts([t for t, y in zip(train_text_int, Ytrain) if y == 0], A0, pi0)
compute_counts([t for t, y in zip(train_text_int, Ytrain) if y == 1], A1, pi1)

sum of each line = 1

In [18]:
# normalize A and pi so they are valid probability matrices
# convince yourself that this is equivalent to the formulas shown before
A0 /= A0.sum(axis=1, keepdims=True)
pi0 /= pi0.sum()

A1 /= A1.sum(axis=1, keepdims=True)
pi1 /= pi1.sum()

In [19]:
# log A and pi since we don't need the actual probs
logA0 = np.log(A0)
logpi0 = np.log(pi0)

logA1 = np.log(A1)
logpi1 = np.log(pi1)

how many samples belong to class 0 and how many class 0

In [20]:
# compute priors
count0 = sum(y == 0 for y in Ytrain)
count1 = sum(y == 1 for y in Ytrain)
total = len(Ytrain)
p0 = count0 / total
p1 = count1 / total
logp0 = np.log(p0)
logp1 = np.log(p1)
p0, p1

(0.3263157894736842, 0.6736842105263158)

constructor - 3 arg
k size of list we pass in

In [21]:
# build a classifier
class Classifier:
  def __init__(self, logAs, logpis, logpriors):
    self.logAs = logAs
    self.logpis = logpis
    self.logpriors = logpriors
    self.K = len(logpriors) # number of classes

  def _compute_log_likelihood(self, input_, class_):
    logA = self.logAs[class_]
    logpi = self.logpis[class_]

    last_idx = None
    logprob = 0
    for idx in input_:
      if last_idx is None:
        # it's the first token
        logprob += logpi[idx]
      else:
        logprob += logA[last_idx, idx]

      # update last_idx
      last_idx = idx

    return logprob

  def predict(self, inputs):
    predictions = np.zeros(len(inputs))
    for i, input_ in enumerate(inputs):
      posteriors = [self._compute_log_likelihood(input_, c) + self.logpriors[c] \
             for c in range(self.K)]
      pred = np.argmax(posteriors)
      predictions[i] = pred
    return predictions

In [22]:
# each array must be in order since classes are assumed to index these lists
clf = Classifier([logA0, logA1], [logpi0, logpi1], [logp0, logp1])

In [23]:
Ptrain = clf.predict(train_text_int)
print(f"Train acc: {np.mean(Ptrain == Ytrain)}")

Train acc: 0.9944272445820433


In [24]:
Ptest = clf.predict(test_text_int)
print(f"Test acc: {np.mean(Ptest == Ytest)}")

Test acc: 0.8014842300556586


In [25]:
from sklearn.metrics import confusion_matrix, f1_score

# read about F-score: https://en.wikipedia.org/wiki/F-score

most of the values fall on diagonal leading to highest accuracy.

In [26]:
cm = confusion_matrix(Ytrain, Ptrain)
cm

array([[ 518,    9],
       [   0, 1088]])

In [27]:
cm_test = confusion_matrix(Ytest, Ptest)
cm_test

array([[ 94,  97],
       [ 10, 338]])

In [28]:
f1_score(Ytrain, Ptrain)

0.9958810068649886

In [29]:
f1_score(Ytest, Ptest)

0.8633461047254151